# Execute a typed tool and recover after a failure

Prerequisites: Function contracts, transactions, capability checks and idempotency.

A tool-using system needs an execution contract independent of the language used to request the action. The companion accepts one schema: record_count with an integer count, a named idempotency key and an explicit capability flag. Unknown tools, malformed arguments and ungranted capabilities fail before an effect is accepted.

The effect and request digest are committed together in an in-memory SQLite transaction. A simulated crash then occurs before the workflow's next checkpoint. Replaying the same request finds the committed digest and returns the existing result instead of duplicating the effect. Reusing the key for different arguments fails. This demonstrates the required state transitions inside one process; the database is not durable across program termination.

Exactly-once behavior cannot be assumed for arbitrary remote tools. A remote payment or message service must support its own idempotency or a suitable reconciliation protocol. The local fixture deliberately performs no network call, shell command or external account change. Attempts are bounded to two in the scripted scenario; a production worker would persist its cumulative attempt budget separately from a rollbackable workflow checkpoint.

## Model structure

| Element | Model | Interpretation |
| --- | --- | --- |
| Validate | Known tool + integer arguments | Reject malformed requests |
| Authorize | Explicit record_count capability | Gate the effect |
| Commit / replay | Key + payload digest + effect | Avoid duplicate accepted effects |

## Predict

Default execution logs effect.committed, checkpoint.restored, effect.replayed. The final effect count is one. Removing the capability must raise PermissionError; replaying a changed count must raise ValueError.

In [ ]:
from pathlib import Path
import sys, math
base=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(base))
import ai_models as m

In [ ]:
result=m.agent()
assert result['effect_count']==1
assert result['events']==['effect.committed','checkpoint.restored','effect.replayed']
print(result)
try:
    m.agent(allowed=False)
except PermissionError:
    print('Denied as expected')
else:
    raise AssertionError('Capability boundary failed')

## Perturb and explain

Add a second tool with a distinct schema and capability. Test invalid arguments, denied permission, same-payload retry and conflicting-payload retry before adding any external side effect.

Record the changed input, prediction, result and a limit of your conclusion. Restart the kernel to reset the experiment.

## Check your reasoning

What makes a replay safe in this fixture?

1. The request sounds reasonable
2. A matching key and payload digest in the effect transaction
3. Calling the tool twice quickly

Answer and explanations: 1: Natural-language plausibility is not an execution guarantee. 2: The stored digest distinguishes a retry from a conflicting new request. 3: Timing does not prevent duplicate effects.

## Primary evidence

- [SQLite — Atomic Commit](https://www.sqlite.org/atomiccommit.html)
- [Yao et al. — ReAct](https://openreview.net/forum?id=WE_vluYUL-X)

Review scope: this authored worked example and its stated assumptions; no hardware or production acceptance is implied.